# 🧠 Text-to-SQL Notebook

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://media.licdn.com/dms/image/v2/D5612AQFGU3VWgw5iog/article-cover_image-shrink_600_2000/article-cover_image-shrink_600_2000/0/1722756895197?e=2147483647&v=beta&t=TPHLJ_UgAQLW3Ig_pbbPInyEsMygI6qsn6brq2hIwgE"> 
</p>
</div>

## Description :
This AI-powered app allows users to interact with any local SQLite database using natural language queries. It uses **Claude** (accessed via **LiteLLM**) to translate plain English questions into optimized SQL queries, executes them on the selected database, and displays the results in a clean, user-friendly interface.

#### ✅ Key Features

- **🔍 Natural Language Querying:** Ask questions like “What are the top 5 customers by sales?” and get real-time answers from your database.

- **🗂️ Table Explorer:** View all tables in your selected SQLite database and their schema instantly.

- **⚡ Smart Query Generation:** Uses Claude’s reasoning abilities to write complex SQL even from vague or ambiguous questions.

- **📦 Local Database Support:** Upload or connect to any `.db` or `.sqlite` file without server dependencies.

- **🛠️ Powered by LiteLLM:** Seamless support for Claude and other LLMs via a lightweight abstraction.



#### 🎯 Use Cases

- Business analysts exploring customer or sales data  

- Developers debugging or querying large SQLite datasets  

- Educational purposes to learn SQL interactively  

- Rapid prototyping with real data using AI


## Step 1: Environment Setup and Installation

This cell handles initial setup for the notebook:

- Installs dependencies from `requirements/text_to_sql.requirements.txt`.

- Retries installation up to 3 times on failure.

- Loads environment variables from `.env` using `python-dotenv`.

- Ensures `OPENAI_API_KEY` is set before continuing.

After setup, it clears the output and confirms success.


In [ ]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output
from dotenv import load_dotenv
import os

requirements_installed = False
max_retries = 3
retries = 0
REQUIRED_ENV_VARS = ["OPENAI_API_KEY"]
PROJECT_NAME = "text_to_sql"
REQUIREMENTS_FILE = f"{PROJECT_NAME}.requirements.txt"


def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed, retries, max_retries
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    install_status = os.system(f"pip install -r requirements/{REQUIREMENTS_FILE}")
    if install_status == 0:
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return


def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True)

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)


install_requirements()
clear_output()
setup_env()
print("🚀 Setup complete. Continue to the next cell.")

## Step 2: SQLite Database Connection Handling

This code provides a robust way to safely access a SQLite database by first unlocking it through a backup and replacement method.

The `unlock_database` function creates a temporary backup of the database, then replaces the original file with this new copy to prevent lock-related issues.

The `create_connection` function calls the unlock method first, then establishes a connection to the SQLite database with a timeout for stability.

Comprehensive exception handling with `traceback` ensures that any database-related errors are clearly logged for easier debugging.



In [ ]:
import sqlite3
import traceback
from typing import Union
import os


def unlock_database(db_file: str) -> None:
    """
    Unlock the database file by creating a new copy.

    Args:
        db_file (str): Path to the database file.
    """
    try:
        print("Unlocking database:", db_file)

        new_db = f"{db_file}.new"

        # Open the original database
        conn = sqlite3.connect(db_file)
        backup_conn = sqlite3.connect(new_db)

        with backup_conn:
            conn.backup(backup_conn)  # Safely backup the database

        conn.close()
        backup_conn.close()

        # Replace the old database with the new copy
        os.replace(new_db, db_file)

        print("Database unlocked successfully.")

    except Exception as e:
        print(f"Error unlocking database: {e}")
        traceback.print_exc()


def create_connection(db_file: str) -> Union[sqlite3.Connection, None]:
    """
    Create a database connection to the SQLite database specified by DB file.

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    try:
        unlock_database(db_file)
        con = sqlite3.connect(db_file, timeout=10)
        return con
    except Exception as e:
        print(f"Error creating connection: {e}")
        traceback.print_exc()
        return None

## Step 3: SQLite Script Execution Utility

This function executes a multi-statement SQL script using an active SQLite connection and logs the script being executed with a descriptive key.

It creates a cursor from the provided connection and runs the script using `executescript`, which can execute multiple SQL statements at once.

After execution, the changes are committed to the database to ensure persistence.

If any error occurs during execution, it logs the error and prints the full traceback for debugging.


In [ ]:
import sqlite3


def execute_script(
    connection: sqlite3.Connection, script: str, script_key: str
) -> None:
    try:
        print("Executing script: ", script_key)
        cursor = connection.cursor()
        cursor.executescript(script)
        connection.commit()
        print("Script executed successfully")
    except Exception as e:
        print(f"Error executing script: {e}")
        traceback.print_exc()

## Step 4: SQLite Bank Database Seeding Script

This script sets up a basic banking schema and populates it with sample data using a transaction-safe, checkpoint-based approach.

It creates three core tables — `Customers`, `Accounts`, and `Transactions` — with appropriate foreign key relationships and integrity constraints.

`SAVEPOINT`s and `RELEASE`s are used to isolate logical sections of the script for easier rollback and debugging during failures.

The script ensures the data is only inserted if it doesn't already exist (`INSERT OR IGNORE`), providing idempotency for repeated executions.


In [ ]:
bank_db = "data/bank.db"

bank_db_connection = create_connection(bank_db)

seed_script = """
-- Connect to SQLite database
PRAGMA foreign_keys = ON;

BEGIN TRANSACTION;

-- Error handling setup
SAVEPOINT start;

-- Create Customers Table
SAVEPOINT customers;
CREATE TABLE IF NOT EXISTS Customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    phone TEXT NOT NULL,
    address TEXT
);
RELEASE customers;

-- Create Accounts Table
SAVEPOINT accounts;
CREATE TABLE IF NOT EXISTS Accounts (
    account_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    account_type TEXT CHECK(account_type IN ('Checking', 'Savings')) NOT NULL,
    balance DECIMAL(10,2) NOT NULL DEFAULT 0.00,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id) ON DELETE CASCADE
);
RELEASE accounts;

-- Create Transactions Table
SAVEPOINT transactions;
CREATE TABLE IF NOT EXISTS Transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    account_id INTEGER NOT NULL,
    transaction_type TEXT CHECK(transaction_type IN ('Deposit', 'Withdrawal', 'Transfer')) NOT NULL,
    amount DECIMAL(10,2) NOT NULL,
    transaction_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (account_id) REFERENCES Accounts(account_id) ON DELETE CASCADE
);
RELEASE transactions;

-- Ensure customers exist before inserting accounts
SAVEPOINT check_customers;
INSERT OR IGNORE INTO Customers (customer_id, name, email, phone, address) VALUES
    (1, 'Alice Johnson', 'alice@example.com', '123-456-7890', '123 Elm Street'),
    (2, 'Bob Smith', 'bob@example.com', '234-567-8901', '456 Oak Street'),
    (3, 'Charlie Brown', 'charlie@example.com', '345-678-9012', '789 Maple Street');
RELEASE check_customers;

-- Insert sample accounts with valid customer IDs
SAVEPOINT insert_accounts;
INSERT OR IGNORE INTO Accounts (customer_id, account_type, balance) VALUES
    (1, 'Checking', 1500.00),
    (1, 'Savings', 3000.00),
    (2, 'Checking', 2000.00),
    (3, 'Savings', 500.00);
RELEASE insert_accounts;

-- Ensure accounts exist before inserting transactions
SAVEPOINT check_accounts;
INSERT OR IGNORE INTO Accounts (account_id, customer_id, account_type, balance) VALUES
    (1, 1, 'Checking', 1500.00),
    (2, 1, 'Savings', 3000.00),
    (3, 2, 'Checking', 2000.00),
    (4, 3, 'Savings', 500.00);
RELEASE check_accounts;

-- Insert sample transactions with valid account IDs
SAVEPOINT insert_transactions;
INSERT OR IGNORE INTO Transactions (account_id, transaction_type, amount) VALUES
    (1, 'Deposit', 500.00),
    (2, 'Withdrawal', 200.00),
    (3, 'Deposit', 1000.00),
    (4, 'Withdrawal', 50.00);
RELEASE insert_transactions;

COMMIT;

-- Error handling
PRAGMA foreign_keys = ON;
"""

execute_script(bank_db_connection, seed_script, "seed_bank_db")

## Step 5: DBoss Class: SQLite Database Interaction via Natural Language

The `DBoss` class facilitates interaction with an SQLite database using natural language queries, utilizing the `litellm` library for SQL query generation.

- **Initialization**: The class is initialized with a database file, and it establishes a connection using the `create_connection` method.

- **Schema Retrieval**: It provides methods to fetch the full database schema or individual table schemas, allowing users to explore the structure of the database.

- **SQL Query Generation**: It uses natural language input from the user, parses it, and generates the corresponding SQL query using a large language model (LLM) (`claude-3-7-sonnet-latest`) through the `generate_sql_query` method.

- **Query Execution**: The `execute_query` method is used to run the generated SQL queries and retrieve results. The results are returned as a list of tuples.

- **Natural Language Queries**: The `run_text_query` method accepts user input in natural language, converts it into SQL, and executes it on the database, making it easier to interact with the database without knowing SQL syntax.


In [ ]:
from typing import List, Any
from litellm import completion
import json


class DBoss:
    """
    DBoss enables you to interact with your SQLite database in natural language.
    """

    DEFAULT_LLM_MODEL = "anthropic/claude-3-7-sonnet-latest"
    DEFAULT_TEMPERATURE = 0.5

    def __init__(self, db_file: str):
        """
        Initialize DBoss with the database file.

        Args:
            db_file (str): Database file
        """
        self.db_file = db_file
        self.connection = create_connection(db_file)

    def execute_query(self, query: str) -> List[Any]:
        """
        Execute a query on the database.

        Args:
            query (str): SQL query

        Returns:
            None
        """
        try:
            cursor = self.connection.cursor()
            cursor.execute(query)
            rows = cursor.fetchall()
            return rows
        except Exception as e:
            print(f"Error executing query: {e}")
            traceback.print_exc()
            return []

    def _get_full_schema(self) -> List[Any]:
        """
        Get the full schema of the database.

        Returns:
            List: List of tuples containing the schema details
        """
        query = "SELECT name FROM sqlite_master WHERE type='table';"
        return self.execute_query(query)

    def get_table_schema(self, table_name: str) -> List[Any]:
        """
        Get the schema of a specific table.

        Args:
            table_name (str): Name of the table

        Returns:
            List: List of tuples containing the schema details
        """
        query = f"PRAGMA table_info({table_name});"
        return self.execute_query(query)

    def get_all_table_schema(self) -> dict:
        """
        Get the schema of all tables in the database.

        Returns:
            dict: Dictionary containing the schema details of all tables
        """
        schema = {}
        tables = self._get_full_schema()
        for table in tables:
            table_name = table[0]
            schema[table_name] = self.get_table_schema(table_name)
        return schema

    def generate_sql_query(self, user_query: str) -> str:
        """
        Generate a SQL query from a user query.

        Args:
            user_query (str): User query

        Returns:
            str: SQL query
        """

        def strip_backticks_and_sql_block(query: str) -> str:
            query = query.replace("`", "").replace("```sql", "").replace("```", "")
            if query.startswith("sql"):
                query = query[3:]
            return query.strip()

        try:
            user_prompt = f"""
            Given the following user query and database schema, generate the corresponding SQL query:
            User Query: {user_query}
            Database Schema: {json.dumps(self.get_all_table_schema())}
            Simply respond with the SQL query that would generate the desired result and nothing else.
            """
            response = completion(
                messages=[{"role": "user", "content": user_prompt}],
                model=self.DEFAULT_LLM_MODEL,
                temperature=self.DEFAULT_TEMPERATURE,
            )
            return strip_backticks_and_sql_block(response.choices[0].message.content)
        except Exception as e:
            print(f"Error generating SQL query: {e}")
            traceback.print_exc()
            return ""

    def run_text_query(self, user_query: str) -> List[Any]:
        """
        Run a text query on the database.

        Args:
            user_query (str): User query

        Returns:
            List: List of tuples containing the query results
        """
        sql_query = self.generate_sql_query(user_query)
        if not sql_query:
            raise Exception("Failed to generate SQL query.")
        return self.execute_query(sql_query)

## Step 6: Simple Querying Example

This example demonstrates how to use the `DBoss` class for querying an SQLite database in natural language.

- The database file `"data/bank.db"` is specified as the input.

- An instance of the `DBoss` class is created by passing the `bank_db` path to the constructor.

- The `dboss` object is now ready to process natural language queries and interact with the database using the methods provided by the `DBoss` class.

In [ ]:
## Simple querying example

bank_db = "data/bank.db"
dboss = DBoss(bank_db)

## Step 7: Simple Query Execution

In this step, a simple SQL query is executed on the database using the `execute_query` method of the `DBoss` class.

- The query `"SELECT * FROM Customers;"` is executed, which retrieves all records from the `Customers` table.

- The result is stored in the `result` variable, which contains the rows fetched from the query.

- The query is printed for reference.

- A loop iterates over the rows in `result` and prints each row to the console, displaying the customer data retrieved from the database.


In [ ]:
# 1. Simple query execution

query = "SELECT * FROM Customers;"
result = dboss.execute_query(query=query)
print(query)

for row in result:
    print(row)

## Step 8: Get the Schema of a Specific Table

In this step, the schema of a specific table, `"Customers"`, is retrieved using the `get_table_schema` method.

- The `get_table_schema` method is called with the `table_name` argument set to `"Customers"`, which fetches the schema details of the table.

- The schema is stored in the `schema` variable, which is a list of tuples containing the column names and other schema details.

- A message indicating the schema retrieval is printed.

- A loop iterates over the `schema` and prints each row, displaying the structure and columns of the `Customers` table.


In [ ]:
# 2. Get the schema of a specific table

table_name = "Customers"

schema = dboss.get_table_schema(table_name)

print(f"\nSchema of table '{table_name}':")

for row in schema:
    print(row)

## Step 9: Get the Schema of All Tables

In this step, the schema of all tables in the database is retrieved using the `get_all_table_schema` method.

- The `get_all_table_schema` method is called, which internally retrieves the schema of each table in the database and returns it as a dictionary, where the keys are table names and the values are their respective schemas.

- The full schema is stored in the `full_schema` variable, which is a dictionary of schemas for all the tables.

- A message indicating the retrieval of the full schema is printed.

- A loop iterates over the `full_schema` dictionary. For each table, it prints the schema of that specific table, including the column details.


In [ ]:
# 3. Get the schema of all tables

full_schema = dboss.get_all_table_schema()

print("\nFull schema of the database:")

for table_name, schema in full_schema.items():
    print(f"\nSchema of table '{table_name}':")
    for row in schema:
        print(row)

## Step 10: Generate SQL Query from User Query

In this step, a SQL query is generated based on the user's natural language query using the `generate_sql_query` method.

- The `generate_sql_query` method is called with the user query `"Get all accounts with savings above 100"`. This method generates the corresponding SQL query by interacting with the database schema.

- The original user query is printed to provide context.

- The generated SQL query, which is a result of the transformation from natural language to SQL, is printed to show what SQL query will be executed in the database.


In [ ]:
query = "Get all accounts with savings above 100"

sql_query = dboss.generate_sql_query(query)
print(f"\n{query}")
print(f"\n{sql_query}")

## Step 11: Execute Text Query and Retrieve Results

This step demonstrates how to run a text-based query on the database using DBoss.

- The `run_text_query` method is called with the user query `"Get all customer names, amount and account details with Savings above 100"`. This method internally generates the corresponding SQL query, executes it, and retrieves the results.

- The original user query is printed for clarity.

- The results of the query execution are printed, showing customer names, account details, and balances for accounts where the savings are above 100.


In [ ]:
bank_db = "data/bank.db"
query = "Get all customer names, amount and account details with Savings above 100"

dboss = DBoss(bank_db)
result = dboss.run_text_query(query)

print(f"\n{query}")
for row in result:
    print(row)

## Conclusion :

---



# Thank You for visiting The Hackers Playbook! 🌐

If you liked this research material;

- [Subscribe to our newsletter.](https://thehackersplaybook.substack.com)

- [Follow us on LinkedIn.](https://www.linkedin.com/company/the-hackers-playbook/)

- [Leave a star on our GitHub.](https://www.github.com/thehackersplaybook)

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
</div>
